In [3]:
import os
import socketio
import torch
from nnsight.intervention.backends.remote import RemoteBackend
from nnsight import LanguageModel


# Pick a model from the list of available models at https://host.edif.ai/status
model_name = "openai-community/gpt2"
model = LanguageModel(model_name)
tokenizer = model.tokenizer



# Feature for which we check the activations(this is not important, unless we want to check the activations on a specific feature)
TARGET_FID = 107006
# prompt for above
activation_prompt = "In the Zoo i saw a animal it  "

# Prompts that should activate the feature, for the feature we want to find
positive_prompts = [
    "I like cats",
    "Cats are so cute and fluffy!",
    "Cats are the best pets in the world.",
    "There is nothing better than cats.",
    "Surely you love cats don't you?",
    "Aren't cats just the most wonderful creatures on Earth?",
]
# generic neutral prompts that are not expected to activate the feature
neutral_prompts = [
    "I ran home from school.",
    "The meeting started at noon.",
]
# other negative can be technically empty, but i added this incase we have clear opposite examples to positive prompts, use generic prompts if not the case.
other_negative = [
    "I walked to the store.",
    "The meeting started at noon.",
]
# words we expect to show up if we boost said feature, this is not actually important, if any those words show up though, it will increase the score, and we can quickly check whether ANY activated the feature.
feature_words = [
    "cat", "cats", "kitten", "kittens", "kitty", "kitties", "feline", "felines",
    "purr", "purring", "meow", "meowing", "whisker", "whiskers", "tabby"
]

probe_prompt = "I went inside "
boost_strength = 50.0
max_new_tokens = 25


In [4]:
from sae_lens import SAE
#import pandas as pd

release = "gpt2-small-attn-out-v5-128k"
sae_id = "blocks.0.hook_attn_out"
sae = SAE.from_pretrained(release, sae_id)

In [ ]:
#method to steer the feature by manually intervening at each step of generation, applying a delta to the activations of the feature we want to steer. This method allows us to see the effect of steering on the generated text in a more granular way, as we can observe how the activations change at each step and how that affects the output.
def steer_with_feature_manual(prompt, feature_id, strength=5.0, max_new_tokens=15):
    #Steer generation by manually generating one token at a time,
    #with intervention applied at each step.
    delta = strength * sae.W_dec[feature_id].detach() # delta is the vector to add to residual stream, strength is just a multiplier to the direction sae.W_dec

#detach removes tensor from tracking
    
    # Initialize
    tokens = model.tokenizer.encode(prompt, return_tensors="pt")[0]  # here we encode the prompt into tokens using the model's tokenizer, and we take the first element of the resulting tensor to get a 1D tensor of token IDs.
    result_tokens = tokens.clone()
    
    for i in range(max_new_tokens):
        # Run forward pass on current sequence
        with model.trace(model.tokenizer.decode(tokens)) as tracer:
            # Apply intervention at layer 0 (blocks.0.hook_attn_out => h[0]) 
            resid = model.transformer.h[0].output[0] # is this the correct way to access the activations at layer 0? we want to access the output of the attention mechanism at layer 0, which is where our SAE is focused. we need to make sure that this line correctly retrieves those activations so that we can apply our intervention.
            resid[:] = resid + delta.unsqueeze(0).unsqueeze(0)
            
            # Get logits for next token
            logits = model.lm_head.output.save()
        
        # Sample next token (greedy: argmax)
        next_token = torch.argmax(logits[0, -1, :]).unsqueeze(0)
        result_tokens = torch.cat([result_tokens, next_token])
        tokens = torch.cat([tokens, next_token])
        
        # Stop if end token
        if next_token.item() == model.tokenizer.eos_token_id:
            break
    
    return model.tokenizer.decode(result_tokens)





print("BASELINE:")
baseline = steer_with_feature_manual(activation_prompt, TARGET_FID, strength=0, max_new_tokens=15)
print(f"Output: {baseline}\n")

print("WITH STEERING:")
steered_pos = steer_with_feature_manual(activation_prompt, TARGET_FID, strength=50, max_new_tokens=15)
print(f"Output: {steered_pos}\n")

print("OPPOSITE STEERING:")
steered_neg = steer_with_feature_manual(activation_prompt, TARGET_FID, strength=-50, max_new_tokens=15)
print(f"Output: {steered_neg}\n")

BASELINE:


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11305.81it/s]


Output: In the Zoo i saw a animal it    was called a "pig".   It was a small,

WITH STEERING:
Output: In the Zoo i saw a animal it  irl)


The best way to find out cat is to find

OPPOSITE STEERING:
Output: In the Zoo i saw a animal it  




I was aaaaaaaa



In [6]:

def get_sae_features_step1(prompt):
    #Get SAE feature activations for a prompt
    with model.trace(prompt):
        h = model.transformer.h[0].output[0].save()
    
    # h is now a saved tensor, move to device and squeeze batch dimension
    h_device = h.squeeze(0).to(sae.device)
    
    with torch.no_grad():
        feats = sae.encode(h_device)  # [tokens, n_features]
    
    return feats.cpu(), model.tokenizer.tokenize(prompt)

# Collect activations for feature across all prompt categories
results_f = {
    "positive": [],
    "neutral": [],
    "other_negative": []
}

print("=" * 70)
print(f"FEATURE {TARGET_FID} ACTIVATION ANALYSIS")
print("=" * 70)

# Process positive prompts
print("\n[POSITIVE PROMPTS]")
for prompt in positive_prompts:
    feats, _ = get_sae_features_step1(prompt)
    max_act = feats[:, TARGET_FID].max().item()
    mean_act = feats[:, TARGET_FID].mean().item()
    results_f["positive"].append({
        "prompt": prompt,
        "max_activation": max_act,
        "mean_activation": mean_act
    })
    print(f"  max={max_act:7.4f}  mean={mean_act:7.4f}  |  {prompt}")

# Process neutral prompts
print("\n[NEUTRAL PROMPTS]")
for prompt in neutral_prompts:
    feats, _ = get_sae_features_step1(prompt)
    max_act = feats[:, TARGET_FID].max().item()
    mean_act = feats[:, TARGET_FID].mean().item()
    results_f["neutral"].append({
        "prompt": prompt,
        "max_activation": max_act,
        "mean_activation": mean_act
    })
    print(f"  max={max_act:7.4f}  mean={mean_act:7.4f}  |  {prompt}")

# Process other-negative prompts
print("\n[OTHER NEGATIVE PROMPTS]")
for prompt in other_negative:
    feats, _ = get_sae_features_step1(prompt)
    max_act = feats[:, TARGET_FID].max().item()
    mean_act = feats[:, TARGET_FID].mean().item()
    results_f["other_negative"].append({
        "prompt": prompt,
        "max_activation": max_act,
        "mean_activation": mean_act
    })
    print(f"  max={max_act:7.4f}  mean={mean_act:7.4f}  |  {prompt}")

# Compute statistics
print("\n" + "=" * 70)
print("STATISTICS")
print("=" * 70)

for category in ["positive", "neutral", "other_negative"]:
    max_acts = [r["max_activation"] for r in results_f[category]]
    mean_acts = [r["mean_activation"] for r in results_f[category]]
    
    print(f"\n{category.upper()}:")
    print(f"  Mean of max activations:  {sum(max_acts) / len(max_acts):.4f}")
    print(f"  Mean of mean activations: {sum(mean_acts) / len(mean_acts):.4f}")
    print(f"  Std of max activations:   {torch.tensor(max_acts).std().item():.4f}")

# Compare across categories
positive_max_mean = sum([r["max_activation"] for r in results_f["positive"]]) / len(results_f["positive"])
neutral_max_mean = sum([r["max_activation"] for r in results_f["neutral"]]) / len(results_f["neutral"])
other_max_mean = sum([r["max_activation"] for r in results_f["other_negative"]]) / len(results_f["other_negative"])

print("\n" + "-" * 70)
print("COMPARISON: Mean of Max Activations")
print("-" * 70)
print(f"  Postive:      {positive_max_mean:.4f}")
print(f"  Neutral:       {neutral_max_mean:.4f}")
print(f"  Other-Neg:     {other_max_mean:.4f}")
print(f"\n  Postive vs Neutral:       +{(positive_max_mean - neutral_max_mean):.4f}")
print(f"  Postive vs Other-Neg:     +{(positive_max_mean - other_max_mean):.4f}")

if positive_max_mean > neutral_max_mean and positive_max_mean > other_max_mean:
    print(f"\n✅ Feature {TARGET_FID} preferentially activates this feature!")
else:
    print(f"\n⚠️  Feature {TARGET_FID} does not strongly prefer this feature.")

FEATURE 107006 ACTIVATION ANALYSIS

[POSITIVE PROMPTS]
  max=14.1108  mean= 4.7036  |  I like cats
  max= 0.0000  mean= 0.0000  |  Cats are so cute and fluffy!
  max= 0.0000  mean= 0.0000  |  Cats are the best pets in the world.
  max=14.4275  mean= 2.0611  |  There is nothing better than cats.
  max=12.9356  mean= 1.4373  |  Surely you love cats don't you?
  max=13.1828  mean= 1.1984  |  Aren't cats just the most wonderful creatures on Earth?

[NEUTRAL PROMPTS]
  max= 0.0000  mean= 0.0000  |  I ran home from school.
  max= 0.0000  mean= 0.0000  |  The meeting started at noon.

[OTHER NEGATIVE PROMPTS]
  max= 0.0000  mean= 0.0000  |  I walked to the store.
  max= 0.0000  mean= 0.0000  |  The meeting started at noon.

STATISTICS

POSITIVE:
  Mean of max activations:  9.1095
  Mean of mean activations: 1.5667
  Std of max activations:   7.0780

NEUTRAL:
  Mean of max activations:  0.0000
  Mean of mean activations: 0.0000
  Std of max activations:   0.0000

OTHER_NEGATIVE:
  Mean of max 

In [ ]:
# Step 2: Feature Discovery via Activation Contrast
# Discover which features activate most specifically on feature prompts

print("\n" + "=" * 70)
print("FEATURE DISCOVERY: Which features activate on our feature?")
print("=" * 70)

# Collect max activations for all features across all prompts
all_prompts_positive = {
    "positive": positive_prompts,
    "neutral": neutral_prompts,
    "other_negative": other_negative
}

# Store activations per category: [n_prompts, n_features]
activations_per_category = {}

for category, prompts in all_prompts_positive.items():
    category_acts = []
    for prompt in prompts:
        feats, _ = get_sae_features_step1(prompt)  # Fixed: was get_sae_features
        # Take max activation across tokens for each feature
        max_acts = feats.max(dim=0).values  # [n_features]
        category_acts.append(max_acts)
    activations_per_category[category] = torch.stack(category_acts)  # [n_prompts, n_features]

print(f"\nActivations shape per category:")
for cat, acts in activations_per_category.items():
    print(f"  {cat:15s}: {acts.shape}")

# Compute mean activation per category
mean_acts_positive = activations_per_category["positive"].mean(dim=0)  # [n_features]
mean_acts_neutral = activations_per_category["neutral"].mean(dim=0)
mean_acts_other = activations_per_category["other_negative"].mean(dim=0)

print(f"\n✓  Computed mean activations across categories")

# Define specificity score:
# A feature is specific to feature if it activates strongly on feature
# but weakly on neutral and other-negative prompts
# specificity = mean_feature - max(mean_neutral, mean_other_neg)

baseline = torch.maximum(mean_acts_neutral, mean_acts_other)
specificity = mean_acts_positive - baseline  # [n_features]

# Get top features by specificity
top_k = 25
top_feature_indices = torch.argsort(specificity, descending=True)[:top_k]

print(f"\nTop {top_k} features by specificity:")
print(f"{'Rank':<6} {'Feature':>8} {'Specificity':>12} {'P_mean':>10} {'N_mean':>10} {'O_mean':>10}")
print("-" * 65)

for rank, fidx in enumerate(top_feature_indices, 1):
    fidx = fidx.item()
    spec = specificity[fidx].item()
    p_mean = mean_acts_positive[fidx].item()
    n_mean = mean_acts_neutral[fidx].item()
    o_mean = mean_acts_other[fidx].item()
    
    marker = " ◀ TARGET!" if fidx == 18699 else ""
    print(f"{rank:<6} {fidx:>8} {spec:>12.4f} {p_mean:>10.4f} {n_mean:>10.4f} {o_mean:>10.4f}{marker}")

# Check where feature 18699 ranks
# if 18699 in top_feature_indices:
#     rank_18699 = (top_feature_indices == 18699).nonzero(as_tuple=True)[0].item() + 1
#     print(f"\n✅ Feature 18699 is in top-{top_k} at rank #{rank_18699}!")
# else:
#     all_sorted = torch.argsort(specificity, descending=True)
#     rank_18699 = (all_sorted == 18699).nonzero(as_tuple=True)[0].item() + 1
#     spec_18699 = specificity[18699].item()
#     print(f"\n⚠️  Feature 18699 ranked #{rank_18699} (specificity={spec_18699:.4f})")

print(f"\n" + "=" * 70)
print("Analysis complete! Features ranked by specificity.")
print("=" * 70)


FEATURE DISCOVERY: Which features activate on our feature?

Activations shape per category:
  positive       : torch.Size([6, 131072])
  neutral        : torch.Size([2, 131072])
  other_negative : torch.Size([2, 131072])

✓  Computed mean activations across categories

Top 25 features by specificity:
Rank    Feature  Specificity     J_mean     N_mean     O_mean
-----------------------------------------------------------------
1         77740      10.7379    10.7379     0.0000     0.0000
2        107006       9.1095     9.1095     0.0000     0.0000
3         50345       8.9699    17.7829     8.3511     8.8130
4         18183       8.7725     8.7725     0.0000     0.0000
5         24597       8.6865    11.5991     2.9126     0.0000
6         16560       6.7829     6.7829     0.0000     0.0000
7         48513       6.4590     6.4590     0.0000     0.0000
8         20832       6.4378     8.0473     0.0000     1.6095
9         85362       6.4179     6.4179     0.0000     0.0000
10        5

In [8]:
# Blind top-10 causal test: can any of the strongest features push a neutral prompt toward the feature?
# This does not assume a known target feature. It scores generated text for feature-related language.

# probe_prompt = "I saw a "
# #probe_prompt = "In the Zoo i saw a animal. It "
# boost_strength = 50.0
# max_new_tokens = 25

if "top_feature_indices" in globals():
    candidate_features = top_feature_indices[:10].tolist()
else:
    candidate_features = torch.argsort(specificity, descending=True)[:10].tolist()


# calculates score... not necessary, just for convenience
def text_score(text):
    lowered = text.lower()
    return sum(word in lowered for word in feature_words)


def generate_with_feature(prompt, feature_id, strength=0.0, steps=25):
    return steer_with_feature_manual(
        prompt, feature_id, strength=strength, max_new_tokens=steps
    )

print("=" * 70)
print("BLIND TOP-10 FEATURE STEERING TEST")
print("=" * 70)
print(f"Probe prompt: {probe_prompt!r}")
print(f"Boost strength: {boost_strength}")
print(f"Max new tokens: {max_new_tokens}")
print()

baseline = generate_with_feature(probe_prompt, candidate_features[0], strength=0.0, steps=max_new_tokens)
baseline_score = text_score(baseline)
print("BASELINE OUTPUT:")
print(baseline)
print(f"score={baseline_score}")
print()

results = []
for fid in candidate_features:
    output = generate_with_feature(probe_prompt, fid, strength=boost_strength, steps=max_new_tokens)
    current_score = text_score(output)
    results.append((fid, current_score, output))
    print(f"feature {fid:6d} | score={current_score:2d}")
    print(output)
    print("-" * 70)

best = max(results, key=lambda item: item[1])
print()
print(f"Best score: feature {best[0]} with score={best[1]}")
if best[1] > baseline_score:
    print("At least one top-10 feature increased related language relative to baseline.")
else:
    print("None of the top-10 features clearly shifted the generation toward the feature on this probe.")

BLIND TOP-10 FEATURE STEERING TEST
Probe prompt: 'I went inside '
Boost strength: 50.0
Max new tokens: 25

BASELINE OUTPUT:
I went inside   and found a large, dark room with a large, dark ceiling. I was surprised to find that the room was very
score=0

feature  77740 | score= 0
I went inside !"


"I'm so cute, I'm so cute, I'm so…'


'

----------------------------------------------------------------------
feature 107006 | score= 0
I went inside .")


The first person in the house, the first person in the dog, the first person in the dog,
----------------------------------------------------------------------
feature  50345 | score= 0
I went inside  a a a a a a a a a a a a a a a a a a a a a a a a a
----------------------------------------------------------------------
feature  18183 | score= 0
I went inside ))),,,.
.
.a.a.a.a.a.a.a.
----------------------------------------------------------------------
feature  24597 | score= 0
I went inside !"


"
"
"
"
"
"
"
"
"
"


------------------